## ERA5 Wind Data

## Download ERA5 Wind data using CDSAPI

In [ ]:
import cdsapi
import os

c = cdsapi.Client()

In [ ]:
# Output folder
ROOT_OUT = r"D:/CDSAPI_ERA5/Test_Notebook"   # editable path folder

# Area of Interest
AREA = [
    -4.8,   # North
    106.1,  # West
    -6.1,   # South
    107.8   # East
]

YEAR = "2025"

TIME = ["00:00", "06:00", "12:00", "18:00"]  # 6 hourly

In [ ]:
# Download u10

out_file = os.path.join(ROOT_OUT, "u10_2025.grib") # editable file name

print("Starting download: U10...")

c.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "variable": "10m_u_component_of_wind",
        "year": YEAR,
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": TIME,
        "area": AREA,
        "format": "grib"
    },
    out_file
)

print(f"Download completed: {out_file}")

In [ ]:
# Download v10

out_file = os.path.join(ROOT_OUT, "v10_2025.grib")  # editable file name

print("Starting download: V10...")

c.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "variable": "10m_v_component_of_wind",
        "year": YEAR,
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": TIME,
        "area": AREA,
        "format": "grib"
    },
    out_file
)

print(f"Download completed: {out_file}")

Note:
Full multi-year download can be automated using loops

## Load GRIB file data
Load and explore dataset (dimension, veriables, values)

In [ ]:
import xarray as xr

file_u = r"D:/CDSAPI_ERA5/Test_Notebook/u10_2025.grib"
file_v = r"D:/CDSAPI_ERA5/Test_Notebook/v10_2025.grib"

ds_u = xr.open_dataset(file_u, engine="cfgrib")
ds_v = xr.open_dataset(file_v, engine="cfgrib")

In [ ]:
# Dataset Info

print(ds_u)
print(ds_v)

In [ ]:
# Data Variables

print(ds_u.data_vars)
print(ds_v.data_vars)

In [ ]:
# Data Dimensions

print(ds_u.dims)
print(ds_v.dims)

In [ ]:
# Data Shapes

print(ds_u['u10'].shape)
print(ds_v['v10'].shape)

In [ ]:
# Data Coordinates

print(ds_u.coords)
print(ds_v.coords)

In [ ]:
# Range data values

print(ds_u['u10'].min().values, ds_u['u10'].max().values)
print(ds_v['v10'].min().values, ds_v['v10'].max().values)

In [ ]:
# timestep values

print(ds_u['time'].values[0])
print(ds_v['time'].values[0])

## Wind data Visualization

In [ ]:
# Timestep extract index

TIME_INDEX = 0

In [ ]:
# Extract per-timestep data

u10 = ds_u['u10']
v10 = ds_v['v10']

u = u10.isel(time=TIME_INDEX)
v = v10.isel(time=TIME_INDEX)

lon = ds_u['longitude']
lat = ds_u['latitude']

print(u.shape)
print(v.shape)

# shape should be 2D (lat,lon) for quiver plot

In [ ]:
# Plot

import numpy as np
import matplotlib.pyplot as plt

lon = ds_u['longitude']
lat = ds_u['latitude']

lon2d, lat2d = np.meshgrid(lon, lat)

speed = np.sqrt(u**2 + v**2)

plt.figure(figsize=(10,6))

plt.contourf(lon2d, lat2d, speed, levels=20)
plt.colorbar(label="Wind Speed (m/s)")

plt.quiver(lon2d, lat2d, u, v)

selected_time = str(u['time'].values)

plt.title(f"ERA5 Wind Vector\n{selected_time}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

## Conclusion